In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from ISLP import load_data
from ISLP.models import ModelSpec as MS, summarize
from scipy.ndimage import variance
from statsmodels.stats.outliers_influence import variance_inflation_factor

### Load Data

In [3]:
boston = load_data("Boston")

In [4]:
boston.head(2)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6


### Building a Model Matrix

In [5]:
y = boston["medv"]
X = MS(["lstat", "age"])
X = X.fit_transform(boston)
model1 = sm.OLS(y,X)
result1 = model1.fit()
summarize(result1)

,coef,std err,t,P>|t|
intercept,33.2228,0.731,45.458,0.000
lstat,-1.0321,0.048,-21.416,0.000
age,0.0345,0.012,2.826,0.005


### If i want to do apply regression to whole predictors sets except one

In [6]:
boston.head(2)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6


In [7]:
exc = boston.drop(["medv", "age"],axis=1)

In [8]:
exc.head(2)

,crim,zn,indus,chas,nox,rm,dis,rad,tax,ptratio,lstat
0,0.00632,18.0,2.31,0,0.538,6.575,4.0900,1,296,15.3,4.98
1,0.02731,0.0,7.07,0,0.469,6.421,4.9671,2,242,17.8,9.14


In [9]:
X_exc = MS(exc).fit_transform(boston)
model2 = sm.OLS(y, X_exc)
result2 = model2.fit()
summarize(result2)

,coef,std err,t,P>|t|
intercept,41.5251,4.920,8.441,0.000
crim,-0.1214,0.033,-3.683,0.000
zn,0.0465,0.014,3.379,0.001
indus,0.0135,0.062,0.217,0.829
chas,2.8528,0.868,3.287,0.001
nox,-18.4851,3.714,-4.978,0.000
rm,3.6811,0.411,8.951,0.000
dis,-1.5068,0.193,-7.825,0.000
rad,0.2879,0.067,4.322,0.000
tax,-0.0127,0.004,-3.333,0.001


In [10]:
dir(result1)

['HC0_se',
 'HC1_se',
 'HC2_se',
 'HC3_se',
 '_HCCM',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abat_diagonal',
 '_cache',
 '_data_attr',
 '_data_in_cache',
 '_get_robustcov_results',
 '_get_wald_nonlinear',
 '_is_nested',
 '_transform_predict_exog',
 '_use_t',
 '_wexog_singular_values',
 'aic',
 'bic',
 'bse',
 'centered_tss',
 'compare_f_test',
 'compare_lm_test',
 'compare_lr_test',
 'condition_number',
 'conf_int',
 'conf_int_el',
 'cov_HC0',
 'cov_HC1',
 'cov_HC2',
 'cov_HC3',
 'cov_kwds',
 'cov_params',
 'cov_type',
 'df_model',
 'df_resid',
 'diagn',
 'eigenvals',
 'el_test',
 'ess',
 'f_pvalue',
 'f_test',
 'fittedvalues',
 'fvalue',
 'get_influence',
 

In [11]:
result1.rsquared # R_Squared

np.float64(0.5512689379421002)

In [12]:
np.sqrt(result1.scale) # RSE

np.float64(6.173136281359115)

### Finding the variance_inflation_factor() i.e VIF()

In [13]:
X.head(2)

,intercept,lstat,age
0,1.0,4.98,65.2
1,1.0,9.14,78.9


In [14]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF

In [15]:
values = [VIF(X,i) for i in range(1,X.shape[1])]
vif = pd.DataFrame({"vif": values},
                   index=[X.columns[1:]])
vif

,vif
lstat,1.569395
age,1.569395


In [79]:
# upper code is the list comprehension of:
vals = []
for i in range(1, X.values.shape[1]):
    vals.append(VIF(X.values,i))

In [80]:
vals

[np.float64(1.569394800568959), np.float64(1.569394800568959)]

In [74]:
valu = [VIF(X_exc, i) for i in range(1, X_exc.shape[1])]
vif1 = pd.DataFrame({"VIF": valu},
                    index=[X_exc.columns[1:]])

In [75]:
vif1

,VIF
crim,1.767455
zn,2.265259
indus,3.987176
chas,1.068018
nox,4.070020
rm,1.834792
dis,3.613722
rad,7.396707
tax,8.994939
ptratio,1.785403


## Interaction Term

In [81]:
X = MS(["lstat", "age", ("lstat", "age")]).fit_transform(boston)
model3 = sm.OLS(y,X)
result3 = model3.fit()
summarize(result3)

,coef,std err,t,P>|t|
intercept,36.0885,1.470,24.553,0.000
lstat,-1.3921,0.167,-8.313,0.000
age,-0.0007,0.020,-0.036,0.971
lstat:age,0.0042,0.002,2.244,0.025
